# 1. Setup Environment and Imports
Install necessary packages and import standard libraries such as `torch`, `torch.nn`, `torch.nn.functional`, `pandas`, `matplotlib.pyplot`, and `huggingface_hub`.

In [ ]:
# !pip install -q torch pandas matplotlib huggingface_hub

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

# 2. Define SAE Architecture Classes
Implement the `BatchTopKSAE` PyTorch module provided in the research documentation to handle the encoding, thresholding, and decoding steps.

In [ ]:
class BatchTopKSAE(nn.Module):
    def __init__(self, d_in, d_sae, k):
        super().__init__()
        self.k       = k
        self.b_pre   = nn.Parameter(torch.zeros(d_in))
        self.encoder = nn.Linear(d_in, d_sae, bias=True)
        self.decoder = nn.Linear(d_sae, d_in, bias=True)

    def encode(self, x):
        x_centered = x - self.b_pre
        pre_acts   = self.encoder(x_centered)
        n_keep     = int(pre_acts.numel() * self.k / pre_acts.shape[-1])
        threshold  = pre_acts.reshape(-1).topk(n_keep).values.min()
        acts       = pre_acts * (pre_acts >= threshold).float()
        return F.relu(acts)

    def decode(self, z):
        return self.decoder(z) + self.b_pre

    def forward(self, x):
        z     = self.encode(x)
        recon = self.decode(z)
        return recon, z

# 3. Download and Load Grid Search Results
Use `hf_hub_download` to fetch the `results.csv` file from `jakelipner/sae-grid-search-layer12` and load it into a pandas DataFrame.

In [ ]:
csv_path = hf_hub_download(
    repo_id="jakelipner/sae-grid-search-layer12",
    filename="results.csv",
    repo_type="model"
)

results_df = pd.read_csv(csv_path)
results_df.head()

# 4. Visualize Hyperparameter Sweep
Generate scatter and line plots comparing validation MSE, NMSE, L0, and dead feature percentages across different Expansion and $K$ parameters for both BatchTopK and TopK types.

In [ ]:
import seaborn as sns

if 'Dead %' in results_df.columns and results_df['Dead %'].dtype == object:
    results_df['Dead %'] = results_df['Dead %'].str.rstrip('%').astype('float') / 100.0

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

sns.lineplot(data=results_df, x='K', y='Val MSE', hue='Expansion', style='Type', markers=True, ax=axes[0])
axes[0].set_title('Val MSE vs K')

sns.lineplot(data=results_df, x='K', y='NMSE', hue='Expansion', style='Type', markers=True, ax=axes[1])
axes[1].set_title('NMSE vs K')

sns.lineplot(data=results_df, x='K', y='L0', hue='Expansion', style='Type', markers=True, ax=axes[2])
axes[2].set_title('L0 vs K')

sns.lineplot(data=results_df, x='K', y='Dead %', hue='Expansion', style='Type', markers=True, ax=axes[3])
axes[3].set_title('Dead Feature % vs K')

plt.tight_layout()
plt.show()

# 5. Load Specific SAE Checkpoint
Download a specific checkpoint, such as `BatchTopK_exp8x_k220.pt`, instantiate the `BatchTopKSAE` model with $d_{in}=896$, $d_{sae}=7168$, and $K=220$, and load the state dictionary.

In [ ]:
path = hf_hub_download(
    repo_id="jakelipner/sae-grid-search-layer12",
    filename="BatchTopK_exp8x_k220.pt",  # change as needed
    repo_type="model"
)

sae = BatchTopKSAE(d_in=896, d_sae=7168, k=220)
sae.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
sae.eval()
print("SAE model loaded successfully.")

# 6. Evaluate SAE Forward Pass
Create test or dummy activation data $x$ and run it through the loaded SAE model to inspect the reconstructed activations $\hat{x}$ and the latent features $z$.

In [ ]:
# Create dummy activation data representing base activations (OhhMoo instruct_base layer 12)
N = 512
d_in = 896
x_dummy = torch.randn(N, d_in)

with torch.no_grad():
    x_hat, z = sae(x_dummy)

print(f"Input shape: {x_dummy.shape}")
print(f"Reconstructed shape: {x_hat.shape}")
print(f"Latent shape: {z.shape}")

l0_norm = (z > 0).float().sum(dim=-1).mean().item()
print(f"Average L0 Norm on dummy data: {l0_norm:.2f}")

# Summary\nThis notebook has successfully implemented the analysis architecture from Michael_2's work, but using the `jakelipner/sae-grid-search-layer12` repository and the `BatchTopKSAE` module as specified.

# 7. Activation-Based Correlation and Capability Mapping\nImplementing validation steps based on feature correlation and logit uplift mapping.

In [ ]:
def compute_activation_correlation(activations_A: torch.Tensor, activations_B: torch.Tensor):\n    \"\"\"\n    Computes the Pearson correlation matrix between feature activations from two checkpoints.\n    activations_A, activations_B: tensors of shape [batch_size, seq_len, num_features]\n    Returns shape: [num_features, num_features]\n    \"\"\"\n    A_flat = activations_A.view(-1, activations_A.shape[-1])\n    B_flat = activations_B.view(-1, activations_B.shape[-1])\n    \n    A_mean = A_flat.mean(dim=0, keepdim=True)\n    B_mean = B_flat.mean(dim=0, keepdim=True)\n    A_centered = A_flat - A_mean\n    B_centered = B_flat - B_mean\n    \n    A_norm = torch.norm(A_centered, dim=0, keepdim=True) + 1e-8\n    B_norm = torch.norm(B_centered, dim=0, keepdim=True) + 1e-8\n    \n    correlation_matrix = torch.matmul(A_centered.T, B_centered) / (A_norm.T * B_norm)\n    return correlation_matrix\n\ndef compute_logit_uplift(feature_decoder_weights: torch.Tensor, unembedding_matrix: torch.Tensor, top_k: int = 10):\n    \"\"\"\n    Maps structural feature births to capabilities via Logit Uplift.\n    Calculates decoder weight projection into the vocabulary space to identify what tokens the feature promotes.\n    \n    feature_decoder_weights: [num_features, d_model]\n    unembedding_matrix: [d_model, vocab_size]\n    \"\"\"\n    logit_contributions = torch.matmul(feature_decoder_weights, unembedding_matrix)\n    top_logits, top_indices = torch.topk(logit_contributions, k=top_k, dim=-1)\n    return top_logits, top_indices

# 8. Feature Birth/Death Tracking\nMetrics and structures for matching SAE variants structurally through Hungarian alignment over correlation matrices.

In [ ]:
from scipy.optimize import linear_sum_assignment\nimport numpy as np\n\nBIRTH_DEATH_THRESHOLD = 0.7\n\ndef match_features_hungarian(sim_matrix, threshold):\n    \"\"\"\n    Matches features across two checkpoints based on a similarity matrix.\n    Identifies features that persist, are born, or die based on a threshold.\n    \"\"\"\n    # If sim_matrix is tensor, convert to numpy\n    if isinstance(sim_matrix, torch.Tensor):\n        sim_matrix = sim_matrix.cpu().numpy()\n    \n    cost_matrix = 1.0 - np.abs(sim_matrix)\n    row_ind, col_ind = linear_sum_assignment(cost_matrix)\n    good_matches = np.abs(sim_matrix[row_ind, col_ind]) >= threshold\n    \n    return {\n        \"n_matched\": int(good_matches.sum()),\n        \"n_births\": len(set(range(sim_matrix.shape[1])) - set(col_ind[good_matches])),\n        \"n_deaths\": len(set(range(sim_matrix.shape[0])) - set(row_ind[good_matches]))\n    }

# 9. Interpretability (Neuronpedia-Style Context Extraction)\nFinding contexts that most highly activate surviving/dead features to manually attribute semantics to them.

In [ ]:
def identify_top_activating_contexts(sae, X_val, input_tokens, tokenizer_mock_decode, feature_idx, top_k=3):\n    \"\"\"\n    B.2 Feature Interpretability (Neuronpedia-style)\n    Extracts the top sequences where specific features fire the hardest.\n    \"\"\"\n    sae.eval()\n    with torch.no_grad():\n        try:\n             _, z = sae(X_val) if isinstance(sae(X_val), tuple) else (None, sae(X_val))\n        except:\n             z = sae(X_val)\n             \n        feature_activations = z[..., feature_idx] # [batch, seq_len]\n        \n        # Find top-k activating positions\n        flat_acts = feature_activations.flatten()\n        top_k = min(top_k, flat_acts.numel())\n        top_values, top_indices = torch.topk(flat_acts, top_k)\n        \n        results = []\n        for idx in top_indices:\n            b = idx.item() // X_val.shape[1]\n            s = idx.item() % X_val.shape[1]\n            \n            # Get surrounding context (e.g., window of 10 tokens)\n            start_context = max(0, s - 5)\n            end_context = min(X_val.shape[1], s + 2)\n            \n            context_tokens = input_tokens[b, start_context:end_context].tolist()\n            context_string = tokenizer_mock_decode(context_tokens)\n            trigger_string = tokenizer_mock_decode([input_tokens[b, s].item()])\n            \n            results.append({\n                \"activation_value\": round(flat_acts[idx].item(), 4),\n                \"context\": context_string,\n                \"trigger_token\": trigger_string\n            })\n    return results

# 10. Evaluating Validation Metrics Pipeline\nCalculating actual Normalized MSE, L0 Norm, and Delta Loss proxies.

In [ ]:
def evaluate_sae_metrics(sae, X_val, original_loss_fn, model_forward_fn):\n    \"\"\"\n    A. Core SAE Evaluation Metrics\n    \"\"\"\n    sae.eval()\n    with torch.no_grad():\n        try:\n            out = sae(X_val)\n            if isinstance(out, tuple):\n                X_reconstructed, z = out[0], out[1]\n            else:\n                X_reconstructed, z = out, out\n        except:\n            X_reconstructed = sae(X_val)\n            z = X_reconstructed \n            \n        # 1. Reconstruction Loss (MSE)\n        mse_loss = F.mse_loss(X_reconstructed, X_val).item()\n        variance = torch.var(X_val).item() if torch.var(X_val).item() > 0 else 1.0\n        nmse = mse_loss / variance\n\n        # 2. Average L0 Norm\n        l0_norm = (z > 0).float().sum(dim=-1).mean().item()\n        \n        # 3. Model Delta Loss (Placeholder)\n        delta_loss_placeholder = 0.0 \n\n    return {\n        \"NMSE\": nmse,\n        \"L0_Norm\": l0_norm,\n        \"Delta_Loss\": delta_loss_placeholder\n    }

# 11. Representation Geometry (Linear CKA)
Adapting the CKA (Centered Kernel Alignment) logic from `Michael/analysis.ipynb` to compare the global representational space of different SAE hyperparameters (e.g. comparing BatchTopK vs TopK).

In [ ]:
def linear_CKA(X, Y):\n    \"\"\"\n    Compute Linear Centered Kernel Alignment between two representation\n    matrices X and Y. For SAE weights, treating them as features x representations.\n    \"\"\"\n    if X.shape != Y.shape:\n        return 0.0\n\n    # Center columns\n    X = X - X.mean(axis=0)\n    Y = Y - Y.mean(axis=0)\n\n    XTX = X.T @ X\n    YTY = Y.T @ Y\n    YTX = Y.T @ X\n\n    num = np.linalg.norm(YTX, \"fro\") ** 2\n    denom = np.linalg.norm(XTX, \"fro\") * np.linalg.norm(YTY, \"fro\")\n\n    if denom == 0:\n        return 0.0\n    return float(num / denom)\n\n# Mock CKA test\ndummy_weights_A = np.random.randn(896, 7168)\ndummy_weights_B = np.random.randn(896, 7168)\nprint(f\"Mock CKA Alignment between variants: {linear_CKA(dummy_weights_A, dummy_weights_B):.4f}\")

# 12. Weight Norm Distributions (Sparsity Proxies)
Plotting the KDE of the decoder feature norms. In previous work, we tracked this over time. Here we can track it to compare `K=32` vs `K=220`.

In [ ]:
def plot_weight_norm_distributions(decoder_weights_dict):\n    \"\"\"\n    Plots the distribution of decoder L2 norms (e.g. comparing configs instead of steps).\n    decoder_weights_dict: dict of label -> torch.Tensor shape [d_model, d_sae]\n    \"\"\"\n    plt.figure(figsize=(10, 6))\n    colors = sns.color_palette(\"viridis\", n_colors=len(decoder_weights_dict))\n    \n    for i, (label, w_dec) in enumerate(decoder_weights_dict.items()):            \n        norms = torch.norm(w_dec.float(), p=2, dim=0)\n        sns.kdeplot(norms.cpu().numpy(), color=colors[i], label=label, alpha=0.7)\n\n    plt.title(\"Decoder Feature Norm Distribution\")\n    plt.xlabel(\"L2 Norm\")\n    plt.ylabel(\"Density\")\n    plt.legend()\n    plt.tight_layout()\n    plt.show()

# 13. Polysemanticity Surrogate (Subspace Variance)
Off-diagonal mean cosine similarity to estimate geometric feature entanglement changes between variants.

In [ ]:
def compute_mutual_orthogonality(w_dec, num_samples=2000):\n    \"\"\"\n    Computes pairwise cosine similarity amongst decoder features\n    to estimate feature entanglement and orthogonality changes.\n    \"\"\"\n    w_dec = w_dec.float()\n    \n    # Normalize features\n    w_norm = w_dec / torch.norm(w_dec, p=2, dim=0, keepdim=True) \n    # W_dec shape is usually [d_model, d_sae] so features are columns. \n    w_norm = w_norm.T # we want [d_sae, d_model] for pairwise dot product\n    \n    # We sample a subset to avoid huge matrix multiplications out of memory\n    num_samples = min(num_samples, w_norm.shape[0])\n    indices = torch.randperm(w_norm.shape[0])[:num_samples]\n    w_sample = w_norm[indices]\n    \n    # Compute cosine similarity matrix\n    sim_matrix = torch.matmul(w_sample, w_sample.T)\n    \n    # We only care about off-diagonal (i != j) mean magnitudes\n    sim_matrix.fill_diagonal_(0)\n    mean_ortho = torch.mean(torch.abs(sim_matrix)).item()\n    return mean_ortho\n\n# Quick test\ndummy_dec = torch.randn(896, 7168)\nprint(f\"Mock Orthogonality: {compute_mutual_orthogonality(dummy_dec):.4f}\")